# Qwen3-8B 4-bit QLoRA DDP 혼합 SFT

기본값인 `TRAINING_STRATEGY="mixed"`는 민사법·지식재산권법·행정법·형사법 데이터를 하나로 합쳐 매 epoch마다 섞어서 학습합니다.

- 대상: Kaggle NVIDIA Tesla T4 16GB ×2, DDP
- 학습: `unsloth/Qwen3-8B-unsloth-bnb-4bit` + 4-bit QLoRA
- 배포: 최종 LoRA를 베이스 모델에 병합한 Hugging Face `merged_16bit` 모델
- 데이터: `data/training_labels_random_10000_each_seed20260925` 아래 4개 폴더
- 권장 기본값: 4개 분야 혼합 학습(분야별 표본 수가 거의 같아 별도 가중치 없이 전역 shuffle)
- 결과: LoRA adapter + 전략별 `/kaggle/working/...-merged-16bit` 병합 모델

## 0. 패키지 설치

처음 한 번만 실행한 뒤 커널을 재시작하세요. Windows 네이티브에서 CUDA 패키지 문제가 생기면 WSL2 Ubuntu 환경을 권장합니다.

In [1]:
# # Kaggle Settings에서 Internet을 켜거나, OFFLINE_WHEEL_DIR에 wheel Dataset 경로를 지정하세요.
# import importlib.util
# import socket
# import subprocess
# import sys
# from pathlib import Path

# OFFLINE_WHEEL_DIR = None  # 예: '/kaggle/input/jurisflow-python-wheels'
# PACKAGES = ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets", "accelerate", "bitsandbytes", "tensorboard", "trackio", "cloudpickle"]
# IMPORT_NAMES = {"unsloth_zoo": "unsloth_zoo", "tensorboard": "tensorboard"}
# missing = [p for p in PACKAGES if importlib.util.find_spec(IMPORT_NAMES.get(p, p)) is None]

# if missing:
#     if OFFLINE_WHEEL_DIR:
#         wheel_dir = Path(OFFLINE_WHEEL_DIR)
#         if not wheel_dir.is_dir():
#             raise FileNotFoundError(f"OFFLINE_WHEEL_DIR가 없습니다: {wheel_dir}")
#         command = [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", str(wheel_dir), *missing]
#     else:
#         try:
#             socket.getaddrinfo("pypi.org", 443)
#         except socket.gaierror as exc:
#             raise RuntimeError(
#                 "PyPI DNS 조회 실패입니다. Kaggle 우측 Settings/Session options에서 Internet을 켠 뒤 "
#                 "세션을 재시작하세요. 인터넷 제한 대회라면 Dependency Manager 또는 wheel Dataset을 사용하세요."
#             ) from exc
#         command = [sys.executable, "-m", "pip", "install", "-q", *missing]
#     print("설치 대상:", missing)
#     subprocess.check_call(command)
#     print("설치가 끝났습니다. 이 셀에서 새로 설치했다면 커널을 재시작한 뒤 처음부터 실행하세요.")
# else:
#     print("필수 패키지가 이미 설치되어 있습니다.")


## 1. 설정

In [2]:
from pathlib import Path
import gc
import hashlib
import json
import os
import random
import subprocess
import time

from datasets import Dataset

BASE_MODEL_OVERRIDE = None  # 인터넷이 없으면 Kaggle Input의 로컬 4-bit 모델 폴더 지정
BASE_MODEL = BASE_MODEL_OVERRIDE or "unsloth/Qwen3-8B-unsloth-bnb-4bit"
DATASET_DIR_NAME = "dataset"
TRAINING_STRATEGY = "mixed"  # 권장: "mixed" / 비교용: "sequential"

SEQUENTIAL_STAGES = [
    {"folder": "01_민사법",        "name": "stage_01_civil",          "lr": 1e-4},
    {"folder": "02_지식재산권법", "name": "stage_02_intellectual_ip", "lr": 1e-4},
    {"folder": "03_행정법",       "name": "stage_03_administrative",  "lr": 1e-4},
    {"folder": "04_형사법",       "name": "stage_04_criminal",        "lr": 1e-4},
]

if TRAINING_STRATEGY == "mixed":
    # DATA_ROOT 아래 네 분야 폴더의 JSON을 한 번에 재귀 탐색하고 전역 shuffle합니다.
    STAGES = [{"folder": ".", "name": "mixed_4domains", "lr": 1e-4}]
    OUTPUT_DIR_NAME = "qwen3-8b-jurisflow-qlora-mixed"
    MERGED_OUTPUT_DIR_NAME = "qwen3-8b-jurisflow-qlora-mixed-merged-16bit"
    EXPECTED_JSON_PER_STAGE = 40_000
elif TRAINING_STRATEGY == "sequential":
    STAGES = SEQUENTIAL_STAGES
    OUTPUT_DIR_NAME = "qwen3-8b-jurisflow-qlora-sequential"
    MERGED_OUTPUT_DIR_NAME = "qwen3-8b-jurisflow-qlora-merged-16bit"
    EXPECTED_JSON_PER_STAGE = 10_000
else:
    raise ValueError("TRAINING_STRATEGY는 'mixed' 또는 'sequential'이어야 합니다.")

# Kaggle Dataset slug를 몰라도 /kaggle/input 아래에서 데이터 폴더를 자동 탐색합니다.
PROJECT_ROOT_OVERRIDE = "/kaggle/working" if Path("/kaggle/working").is_dir() else None
DATA_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None

START_STAGE = 1
END_STAGE = len(STAGES)
SKIP_COMPLETED_STAGES = True
RESUME_INTERRUPTED_STAGE = True

MAX_SEQ_LENGTH = 512
NUM_EPOCHS_PER_STAGE = 1
MICRO_BATCH_SIZE = 1
NUM_PROCESSES = 2          # Kaggle T4 x2 DDP
GRADIENT_ACCUMULATION_STEPS = 4  # 1 × 2 GPU × 4 = global batch 8
LORA_R = 16
LORA_ALPHA = 16
VALIDATION_PERMILLE = 10  # 약 1%; 동일 원문(info)은 같은 split에 배정
SEED = 3407
SAVE_STEPS = 250
LOGGING_STEPS = 10

SMOKE_TEST = False       # True이면 단계당 일부 데이터/20 step만 빠르게 확인
SMOKE_TRAIN_ROWS = 256
SMOKE_EVAL_ROWS = 32
SMOKE_MAX_STEPS = 20
RUN_EVAL_AFTER_STAGE = False  # True이면 per-device eval batch 1로 단계별 평가
USE_TRACKIO = False      # True로 바꾸면 Trackio + TensorBoard에 기록
SAVE_MERGED_MODEL = True  # 선택 전략의 전체 학습 완료 후 merged_16bit 모델 생성

assert 1 <= START_STAGE <= END_STAGE <= len(STAGES)
random.seed(SEED)


In [3]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / DATASET_DIR_NAME).is_dir():
            return candidate
    raise FileNotFoundError(
        f"{DATASET_DIR_NAME!r}를 찾지 못했습니다. PROJECT_ROOT_OVERRIDE 또는 DATA_ROOT_OVERRIDE를 지정하세요."
    )

def find_data_root() -> Path:
    if DATA_ROOT_OVERRIDE:
        return Path(DATA_ROOT_OVERRIDE).expanduser().resolve()
    kaggle_input = Path("/kaggle/input/datasets/hwangbh")
    if kaggle_input.is_dir():
        matches = sorted(p for p in kaggle_input.rglob(DATASET_DIR_NAME) if p.is_dir())
        if len(matches) == 1:
            return matches[0]
        if not matches:
            raise FileNotFoundError(f"/kaggle/input에서 {DATASET_DIR_NAME!r}를 찾지 못했습니다.")
        raise RuntimeError(f"동일한 데이터 폴더가 여러 개입니다: {matches}")
    return find_project_root(Path.cwd().resolve()) / "data" / DATASET_DIR_NAME

PROJECT_ROOT = (
    Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    if PROJECT_ROOT_OVERRIDE
    else find_project_root(Path.cwd().resolve())
)
DATA_ROOT = find_data_root()
OUTPUT_ROOT = (
    Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve()
    if OUTPUT_ROOT_OVERRIDE
    else PROJECT_ROOT / "outputs" / (OUTPUT_DIR_NAME + ("_smoke" if SMOKE_TEST else ""))
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_ROOT:    {DATA_ROOT}")
print(f"OUTPUT_ROOT:  {OUTPUT_ROOT}")
gpu_lines = subprocess.run(["nvidia-smi", "-L"], check=True, capture_output=True, text=True).stdout.strip().splitlines()
print("GPU:", *gpu_lines, sep="\n  " )
if len(gpu_lines) < NUM_PROCESSES:
    raise RuntimeError(f"GPU {NUM_PROCESSES}개가 필요하지만 {len(gpu_lines)}개만 감지됐습니다. Kaggle Accelerator를 GPU T4 x2로 설정하세요.")
for i, stage in enumerate(STAGES, 1):
    folder = DATA_ROOT / stage["folder"]
    count = sum(1 for _ in folder.rglob("*.json")) if folder.is_dir() else 0
    print(f"[{i}] {stage['folder']}: JSON {count:,}개")
    if count != EXPECTED_JSON_PER_STAGE:
        print(f"  주의: 기대한 {EXPECTED_JSON_PER_STAGE:,}개와 다릅니다. 실제 발견 수={count:,}")


PROJECT_ROOT: /kaggle/working
DATA_ROOT:    /kaggle/input/datasets/hwangbh/dataset
OUTPUT_ROOT:  /kaggle/working/outputs/qwen3-8b-jurisflow-qlora-mixed
GPU:
  GPU 0: Tesla T4 (UUID: GPU-53d1f7fc-66ce-52bd-d1f1-bb655eddb499)
  GPU 1: Tesla T4 (UUID: GPU-b1df0471-68b9-53a5-005e-a841e4bc5e31)
[1] .: JSON 40,000개


## 2. JSON 로더와 결정적 train/validation 분리

JSON의 `taskinfo` 또는 `label`에서 `instruction`, `input`, `output`을 읽습니다. `input`/`output`이 없는 요약형 항목은 SFT 대화쌍으로 만들 수 없어 제외하고, 중복도 제거합니다.

In [4]:
DEFAULT_SYSTEM = "당신은 대한민국 법률 문서를 정확하고 신중하게 처리하는 법률 AI 어시스턴트입니다."

def as_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    return json.dumps(value, ensure_ascii=False).strip()

def stable_validation_bucket(group_key: str) -> int:
    digest = hashlib.sha256(f"{SEED}|{group_key}".encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % 1000

def load_stage_records(stage_folder: Path):
    json_paths = sorted(stage_folder.rglob("*.json"))
    records, errors = [], []
    seen = set()
    excluded_missing = 0

    for path in json_paths:
        try:
            obj = json.loads(path.read_text(encoding="utf-8-sig"))
            payload = obj.get("taskinfo") or obj.get("label") or {}
            instruction = as_text(payload.get("instruction")) or DEFAULT_SYSTEM
            user_input = as_text(payload.get("input"))
            output = as_text(payload.get("output"))
            if not user_input or not output:
                excluded_missing += 1
                continue

            dedupe_key = hashlib.sha256(
                (instruction + "\0" + user_input + "\0" + output).encode("utf-8")
            ).hexdigest()
            if dedupe_key in seen:
                continue
            seen.add(dedupe_key)

            info = as_text(obj.get("info"))
            group_key = info or path.stem
            domain = path.relative_to(DATA_ROOT).parts[0]
            records.append({
                "messages": [
                    {"role": "system", "content": instruction},
                    {"role": "user", "content": user_input},
                    {"role": "assistant", "content": output},
                ],
                "group_key": group_key,
                "domain": domain,
                "source_file": str(path.relative_to(DATA_ROOT)),
            })
        except Exception as exc:
            errors.append({"file": str(path), "error": repr(exc)})

    train_rows, eval_rows = [], []
    for row in records:
        target = eval_rows if stable_validation_bucket(row["group_key"]) < VALIDATION_PERMILLE else train_rows
        target.append(row)

    domains = sorted({row["domain"] for row in records})
    domain_split = {
        domain: {
            "train": sum(row["domain"] == domain for row in train_rows),
            "validation": sum(row["domain"] == domain for row in eval_rows),
        }
        for domain in domains
    }
    stats = {
        "json_files": len(json_paths),
        "valid_unique": len(records),
        "train": len(train_rows),
        "validation": len(eval_rows),
        "excluded_missing_input_or_output": excluded_missing,
        "duplicates_removed": len(json_paths) - excluded_missing - len(errors) - len(records),
        "errors": len(errors),
        "domain_split": domain_split,
    }
    if not train_rows:
        raise ValueError(f"학습 가능한 레코드가 없습니다: {stage_folder}")
    return train_rows, eval_rows, stats, errors

# 전체 단계의 데이터 상태를 GPU 모델 로딩 전에 확인합니다.
stage_data_summary = {}
for i, stage in enumerate(STAGES, 1):
    train_rows, eval_rows, stats, errors = load_stage_records(DATA_ROOT / stage["folder"])
    stage_data_summary[stage["name"]] = stats
    print(f"[{i}] {stage['folder']}: {stats}")
    if errors:
        print("  첫 오류:", errors[0])
del train_rows, eval_rows, errors


[1] .: {'json_files': 40000, 'valid_unique': 31559, 'train': 31258, 'validation': 301, 'excluded_missing_input_or_output': 8441, 'duplicates_removed': 0, 'errors': 0, 'domain_split': {'01_민사법': {'train': 9343, 'validation': 103}, '02_지식재산권법': {'train': 9405, 'validation': 76}, '03_행정법': {'train': 6639, 'validation': 63}, '04_형사법': {'train': 5871, 'validation': 59}}}


## 3. 모델/어댑터 로드 함수

첫 단계는 새 LoRA 어댑터를 생성합니다. 2~4단계는 직전 단계의 저장된 어댑터를 `PeftModel.from_pretrained(..., is_trainable=True)`로 다시 불러옵니다. `is_trainable=True`가 빠지면 어댑터가 추론 모드로 고정되어 이어서 학습되지 않습니다.

In [5]:
# 실제 모델 import/생성은 학습 함수 안에서 수행합니다.
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

def format_datasets(tokenizer, train_rows, eval_rows):
    def render(batch):
        return {
            "text": [
                tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
                )
                for messages in batch["messages"]
            ]
        }

    train_ds = Dataset.from_list(train_rows).map(render, batched=True, remove_columns=list(train_rows[0].keys()))
    eval_ds = None
    if eval_rows:
        eval_ds = Dataset.from_list(eval_rows).map(render, batched=True, remove_columns=list(eval_rows[0].keys()))
    return train_ds, eval_ds


## 4. T4×2 DDP 4-bit QLoRA 혼합/순차 학습

`torchrun`이 독립 Python 프로세스 2개를 띄우고 각 T4에 4-bit 베이스 모델을 복제해 DDP QLoRA를 수행합니다. `mixed`에서는 네 폴더를 하나의 학습 데이터셋으로 만들어 Trainer의 분산 sampler가 매 epoch 전역 shuffle하며, `sequential`에서는 각 단계가 끝날 때 rank 0이 adapter를 저장하고 다음 단계에서 이어서 학습합니다.

In [6]:
def run_distributed_qlora_training():
    # 반드시 transformers/trl/peft보다 Unsloth를 먼저 import합니다.
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from unsloth.chat_templates import train_on_responses_only
    import torch
    import torch.distributed as dist
    from peft import PeftModel
    from transformers.trainer_utils import get_last_checkpoint
    from trl import SFTConfig, SFTTrainer

    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    global_rank = int(os.environ.get("RANK", "0"))
    torch.cuda.set_device(local_rank)
    is_main = global_rank == 0

    def sync():
        if dist.is_initialized():
            dist.barrier()

    def load_base_qlora():
        return FastLanguageModel.from_pretrained(
            model_name=BASE_MODEL,
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=True,
            load_in_8bit=False,
            dtype=None,
            device_map={"": local_rank},
        )

    def load_fresh_qlora_model():
        model, tokenizer = load_base_qlora()
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_R,
            target_modules=LORA_TARGET_MODULES,
            lora_alpha=LORA_ALPHA,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=SEED,
            use_rslora=False,
            loftq_config=None,
        )
        if is_main:
            model.print_trainable_parameters()
        return model, tokenizer

    def load_saved_adapter_for_training(adapter_dir: Path):
        if not (adapter_dir / "adapter_config.json").is_file():
            raise FileNotFoundError(f"이전 단계 어댑터가 없습니다: {adapter_dir}")
        base_model, tokenizer = load_base_qlora()
        model = PeftModel.from_pretrained(base_model, str(adapter_dir), is_trainable=True)
        FastLanguageModel.for_training(model)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        if trainable == 0:
            raise RuntimeError("불러온 어댑터에 학습 가능한 파라미터가 없습니다.")
        if is_main:
            model.print_trainable_parameters()
        return model, tokenizer

    all_results = []
    for stage_number in range(START_STAGE, END_STAGE + 1):
        stage = STAGES[stage_number - 1]
        stage_dir = OUTPUT_ROOT / stage["name"]
        adapter_dir = stage_dir / "adapter"
        checkpoint_dir = stage_dir / "trainer_checkpoints"
        manifest_path = stage_dir / "stage_manifest.json"
        previous_adapter_dir = None if stage_number == 1 else OUTPUT_ROOT / STAGES[stage_number - 2]["name"] / "adapter"

        if is_main:
            print("\n" + "=" * 88)
            print(f"STAGE {stage_number}/{len(STAGES)}: {stage['folder']}")
            print("=" * 88)
        sync()

        completed = manifest_path.is_file() and (adapter_dir / "adapter_config.json").is_file()
        if SKIP_COMPLETED_STAGES and completed:
            if is_main:
                print(f"완료된 단계를 건너뜁니다: {adapter_dir}")
                all_results.append(json.loads(manifest_path.read_text(encoding="utf-8")))
            sync()
            continue

        stage_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        if stage_number == 1:
            model, tokenizer = load_fresh_qlora_model()
            loaded_from = BASE_MODEL
        else:
            model, tokenizer = load_saved_adapter_for_training(previous_adapter_dir)
            loaded_from = str(previous_adapter_dir)

        train_rows, eval_rows, data_stats, data_errors = load_stage_records(DATA_ROOT / stage["folder"])
        if SMOKE_TEST:
            train_rows = train_rows[:SMOKE_TRAIN_ROWS]
            eval_rows = eval_rows[:SMOKE_EVAL_ROWS]
        train_ds, eval_ds = format_datasets(tokenizer, train_rows, eval_rows)

        report_to = ["tensorboard"] + (["trackio"] if USE_TRACKIO else [])
        training_args = SFTConfig(
            output_dir=str(checkpoint_dir),
            dataset_text_field="text",
            max_length=MAX_SEQ_LENGTH,
            packing=True,
            per_device_train_batch_size=MICRO_BATCH_SIZE,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            num_train_epochs=NUM_EPOCHS_PER_STAGE,
            max_steps=SMOKE_MAX_STEPS if SMOKE_TEST else -1,
            learning_rate=stage["lr"],
            warmup_ratio=0.03,
            lr_scheduler_type="cosine",
            optim="adamw_8bit",
            weight_decay=0.01,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=LOGGING_STEPS,
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            save_total_limit=2,
            eval_strategy="no",
            report_to=report_to,
            run_name=f"{OUTPUT_DIR_NAME}-{stage['name']}",
            seed=SEED,
            data_seed=SEED,
            gradient_checkpointing=True,
            dataloader_num_workers=0,
            ddp_find_unused_parameters=False,
        )
        trainer = SFTTrainer(
            model=model,
            processing_class=tokenizer,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            args=training_args,
        )
        trainer = train_on_responses_only(
            trainer,
            instruction_part="<|im_start|>user\n",
            response_part="<|im_start|>assistant\n",
        )

        last_checkpoint = get_last_checkpoint(str(checkpoint_dir)) if RESUME_INTERRUPTED_STAGE else None
        if is_main and last_checkpoint:
            print(f"중단 지점 재개: {last_checkpoint}")
        started_at = time.time()
        train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
        elapsed_seconds = time.time() - started_at

        adapter_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(adapter_dir))
        trainer.save_metrics("train", train_result.metrics)
        trainer.save_state()
        eval_metrics = None
        if RUN_EVAL_AFTER_STAGE and eval_ds is not None:
            eval_metrics = trainer.evaluate()
            trainer.save_metrics("eval", eval_metrics)
        sync()

        if is_main:
            tokenizer.save_pretrained(str(adapter_dir))
            manifest = {
                "stage_number": stage_number,
                "stage_name": stage["name"],
                "data_folder": stage["folder"],
                "base_model": BASE_MODEL,
                "training_method": f"4bit_qlora_{TRAINING_STRATEGY}",
                "world_size": NUM_PROCESSES,
                "loaded_from": loaded_from,
                "adapter_dir": str(adapter_dir),
                "learning_rate": stage["lr"],
                "epochs": NUM_EPOCHS_PER_STAGE,
                "max_seq_length": MAX_SEQ_LENGTH,
                "elapsed_seconds": elapsed_seconds,
                "data_stats": data_stats,
                "data_errors": data_errors[:10],
                "train_metrics": train_result.metrics,
                "eval_metrics": eval_metrics,
                "smoke_test": SMOKE_TEST,
            }
            manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
            all_results.append(manifest)
            print(f"저장 완료: {adapter_dir}")
            print(f"소요 시간: {elapsed_seconds / 3600:.2f}시간")
        sync()

        del trainer, train_result, model, tokenizer, train_ds, eval_ds, train_rows, eval_rows
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        sync()

    if is_main:
        print("\n모든 지정 단계가 끝났습니다.")
        for result in all_results:
            print(result["stage_name"], "->", result["adapter_dir"])

# notebook_launcher는 NVIDIA에서 fork를 사용하므로, torchao가 import 중 CUDA를
# 초기화하는 최신 Kaggle 환경에서는 'Cannot re-initialize CUDA'가 발생할 수 있습니다.
# 학습 함수를 직렬화한 뒤 torchrun의 깨끗한 독립 프로세스에서 실행합니다.
import cloudpickle
import sys

LAUNCH_DIR = PROJECT_ROOT / ".jurisflow_torchrun"
LAUNCH_DIR.mkdir(parents=True, exist_ok=True)
payload_path = LAUNCH_DIR / "qlora_training_function.pkl"
runner_path = LAUNCH_DIR / "run_qlora_payload.py"

with payload_path.open("wb") as payload_file:
    cloudpickle.dump(run_distributed_qlora_training, payload_file)

runner_path.write_text(
    "import sys\n"
    "import cloudpickle\n"
    "with open(sys.argv[1], 'rb') as payload_file:\n"
    "    training_function = cloudpickle.load(payload_file)\n"
    "training_function()\n",
    encoding="utf-8",
)

launch_env = os.environ.copy()
launch_env.setdefault("TOKENIZERS_PARALLELISM", "false")
launch_command = [
    sys.executable,
    "-m",
    "torch.distributed.run",
    "--standalone",
    f"--nproc_per_node={NUM_PROCESSES}",
    str(runner_path),
    str(payload_path),
]
print("실행 명령:", " ".join(launch_command))
subprocess.run(launch_command, check=True, env=launch_env)


실행 명령: /usr/bin/python3 -m torch.distributed.run --standalone --nproc_per_node=2 /kaggle/working/.jurisflow_torchrun/run_qlora_payload.py /kaggle/working/.jurisflow_torchrun/qlora_training_function.pkl



*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W929 08:56:41.008545276 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!

STAGE 1/1: .
==((====))==  Unsloth 2026.9.12: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
==((====))==  Unsloth 2026.9.12: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = F

Loading weights: 100%|██████████| 399/399 [00:05<00:00, 73.43it/s] 
Unsloth 2026.9.12 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
Unsloth 2026.9.12 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301


...

{'train_runtime': '1.162e+04', 'train_samples_per_second': '1.052', 'train_steps_per_second': '0.132', 'train_loss': '1.057', 'epoch': '1'}


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/qwen3-8b-jurisflow-qlora-mixed/mixed_4domains/adapter/tokenizer_config.json.


저장 완료: /kaggle/working/outputs/qwen3-8b-jurisflow-qlora-mixed/mixed_4domains/adapter
소요 시간: 3.23시간

모든 지정 단계가 끝났습니다.
mixed_4domains -> /kaggle/working/outputs/qwen3-8b-jurisflow-qlora-mixed/mixed_4domains/adapter


[rank0]:[W929 12:25:40.166265108 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


CompletedProcess(args=['/usr/bin/python3', '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', '/kaggle/working/.jurisflow_torchrun/run_qlora_payload.py', '/kaggle/working/.jurisflow_torchrun/qlora_training_function.pkl'], returncode=0)

## 5. 최종 LoRA 검증 및 merged_16bit 배포 모델 저장

선택한 전략의 전체 학습이 완료된 경우 최종 LoRA를 4-bit 베이스와 함께 불러와 추론을 확인하고, 베이스 모델에 병합한 Hugging Face 호환 16-bit 배포 모델을 저장합니다. Smoke test나 일부 단계만 실행한 경우 병합을 건너뜁니다.

In [7]:
# 최종 LoRA를 4-bit 베이스와 함께 다시 불러옵니다.
from unsloth import FastLanguageModel
import torch

FINAL_STAGE = len(STAGES)
final_adapter_dir = OUTPUT_ROOT / STAGES[FINAL_STAGE - 1]["name"] / "adapter"
all_stages_complete = all(
    (OUTPUT_ROOT / stage["name"] / "stage_manifest.json").is_file()
    and (OUTPUT_ROOT / stage["name"] / "adapter" / "adapter_config.json").is_file()
    for stage in STAGES
)

if SAVE_MERGED_MODEL and not SMOKE_TEST and END_STAGE == len(STAGES) and all_stages_complete:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(final_adapter_dir),
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        load_in_8bit=False,
        dtype=None,
        device_map={"": 0},
    )
    FastLanguageModel.for_inference(model)

    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM},
        {"role": "user", "content": "계약 해제와 해지의 차이를 간단히 설명해 주세요."},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt"
    ).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
        )
    print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))

    MERGED_OUTPUT_DIR = PROJECT_ROOT / MERGED_OUTPUT_DIR_NAME
    MERGED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained_merged(
        str(MERGED_OUTPUT_DIR),
        tokenizer,
        save_method="merged_16bit",
        maximum_memory_usage=0.5,
    )
    merge_manifest = {
        "base_model": BASE_MODEL,
        "adapter_dir": str(final_adapter_dir),
        "format": "huggingface_transformers",
        "precision": "float16",
        "save_method": "merged_16bit",
    }
    (MERGED_OUTPUT_DIR / "merge_manifest.json").write_text(
        json.dumps(merge_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"병합 배포 모델 저장 완료: {MERGED_OUTPUT_DIR}")
elif SMOKE_TEST:
    print("SMOKE_TEST에서는 병합 배포 모델 저장을 건너뜁니다.")
else:
    print("선택한 전략의 전체 완료 조건을 충족하지 않아 병합 저장을 건너뜁니다.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.12: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Unsloth 2026.9.12 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


계약 해제는 계약의 당사자 간에 합의된 내용에 따라 계약을 종료하는 행위를 의미합니다. 반면, 계약 해지는 계약의 일방 당사자가 상대방에게 계약을 종료할 의사표시를 하여 계약을 종료하는 행위를 말합니다. 따라서 해제는 상호 합의에 의해 이루어지며, 해지는 일방의 의사표시에 의해 이루어집니다.


config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [01:44<00:00, 26.24s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:58<00:00, 29.67s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/qwen3-8b-jurisflow-qlora-mixed-merged-16bit`
병합 배포 모델 저장 완료: /kaggle/working/qwen3-8b-jurisflow-qlora-mixed-merged-16bit


In [11]:
messages = [
    {"role": "system", "content": DEFAULT_SYSTEM},
    {"role": "user", "content": "불기소처분에 대해 설명해주세요."},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt"
).to(model.device)
with torch.inference_mode():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
    )
print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))

Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


불기소처분은 형사소송법 제241조에 따라, 공소제기가 된 사건에서 검사가 특정한 범죄사실에 대해 증거가 부족하거나, 그 범죄사실에 대한 공소제기의 이익이 없을 경우에 이루어지는 결정입니다. 이는 공소제기의 적법성을 기준으로 하며, 검사의 판단에 따라 이루어지며, 피해자나 제3자에 대한 법적 구제를 위한 절차가 따릅니다.


## 6. Hugging Face Hub 업로드

In [9]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

MODEL_DIR = Path(
    "/kaggle/working/qwen3-8b-jurisflow-qlora-mixed-merged-16bit"
)

REPO_ID = "grant88/qwen3-8b-jurisflow-qlora-mixed-merged-16bit"

if not MODEL_DIR.is_dir():
    raise FileNotFoundError(f"병합 모델 폴더가 없습니다: {MODEL_DIR}")

if not (MODEL_DIR / "config.json").is_file():
    raise FileNotFoundError("config.json이 없습니다.")

weight_files = list(MODEL_DIR.glob("*.safetensors"))
if not weight_files:
    raise FileNotFoundError("safetensors 모델 가중치가 없습니다.")

size_gb = sum(
    path.stat().st_size
    for path in MODEL_DIR.rglob("*")
    if path.is_file()
) / 1024**3

print(f"업로드 대상: {MODEL_DIR}")
print(f"파일 수: {sum(path.is_file() for path in MODEL_DIR.rglob('*'))}")
print(f"전체 크기: {size_gb:.2f} GB")
print("가중치 파일:", [path.name for path in weight_files])

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)

api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=True,       # 검증이 끝나면 Hub에서 public으로 변경 가능
    exist_ok=True,
)

result = api.upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(MODEL_DIR),
    commit_message="Upload merged JurisFlow Qwen3-8B model",
)

print("업로드 완료:", result.repo_url)

업로드 대상: /kaggle/working/qwen3-8b-jurisflow-qlora-mixed-merged-16bit
파일 수: 18
전체 크기: 15.27 GB
가중치 파일: ['model-00001-of-00004.safetensors', 'model-00004-of-00004.safetensors', 'model-00003-of-00004.safetensors', 'model-00002-of-00004.safetensors']


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

업로드 완료: https://huggingface.co/grant88/qwen3-8b-jurisflow-qlora-mixed-merged-16bit


In [10]:
ADAPTER_DIR = Path(
    "/kaggle/working/outputs/"
    "qwen3-8b-jurisflow-qlora-mixed/"
    "mixed_4domains/adapter"
)

ADAPTER_REPO_ID = "grant88/qwen3-8b-jurisflow-qlora-mixed-adapter"

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)

api.create_repo(
    repo_id=ADAPTER_REPO_ID,
    repo_type="model",
    private=True,
    exist_ok=True,
)

api.upload_folder(
    repo_id=ADAPTER_REPO_ID,
    repo_type="model",
    folder_path=str(ADAPTER_DIR),
    commit_message="Upload JurisFlow QLoRA adapter",
)

print(f"https://huggingface.co/{ADAPTER_REPO_ID}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

https://huggingface.co/grant88/qwen3-8b-jurisflow-qlora-mixed-adapter
